# DBTITLE 1,Create a table for each feed

In [0]:
list=spark.sql("select * from stockmarket.raw.feedlinks")
display(list)


# DBTITLE 1,Read the XML file

In [0]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd

# save data in tbales

In [0]:
for row in list.collect():
  print(row['newsfeed'])
  url = row['linkes']
  response = requests.get(url)
  root = ET.fromstring(response.content)

  items = []
  for item in root.findall('.//item'):
    title = item.find('title').text
    pubDate = item.find('pubDate').text
    items.append({'title': title, 'pubDate': pubDate})  

  df = pd.DataFrame(items)
  # display(df)
  raw = df[['pubDate','title']]
  #display(raw)
  spark.createDataFrame(raw).write.mode("append").format("delta").saveAsTable(f"stockmarket.raw.{row['newsfeed']}")
  # print(row['linkes'])
